# ADS 实时版本原件尺寸修复

官方 ZIP 实际 168,969,969 字节，第一次固定读取仅得到截断字节。此版在新的 190MB/360s 限定预算内只重取同一官方 URL 一次，完整哈希后归档 NAS；不读市场结果或拟合。

In [1]:
from pathlib import Path,PurePosixPath
from datetime import datetime,timezone
import hashlib,io,json,shlex,subprocess,tarfile,tempfile,zipfile
from src.data.nas_market_data import _settings,nas_api_call,_ssh
root=Path.cwd();card=root/'research/philadelphia_ads_vintage_source_20261023';plan_path=card/'size_transport_repair_plan.json';plan=json.loads(plan_path.read_text());prior=json.loads((card/'run_state.json').read_text())
assert plan['frozen'] and prior['status']=='failed_oversize_and_90s_incomplete_GET' and not (card/'repair_run_state.json').exists()
rawdir=root/'data/snapshots/philadelphia_ads_vintage_20261023';rawdir.mkdir(parents=True,exist_ok=True);partial=rawdir/'ADS_All_Vintages.zip.partial';source=rawdir/'ADS_All_Vintages.zip'
assert not partial.exists() and not source.exists()
(card/'repair_run_state.json').write_text(json.dumps({'status':'full_ZIP_GET_started','repair_plan_sha256':hashlib.sha256(plan_path.read_bytes()).hexdigest(),'additional_GET_attempts':1,'market_outcome_reads':0,'model_fits':0,'accounts':0},indent=2)+'\n')
with tempfile.TemporaryDirectory(prefix='ads-amended-') as tmp:
 headers=Path(tmp)/'headers.txt';started=datetime.now(timezone.utc).isoformat()
 command=['curl','--http1.1','--noproxy','*','--connect-timeout','10','--max-time','360','--fail-with-body','-sS','-L','-D',str(headers),'-o',str(partial),'-H','Accept: application/zip','-A','SOXL-Qlib-source-audit/1.0',plan['exact_URL']]
 completed=subprocess.run(command,capture_output=True,text=True,timeout=370);received=datetime.now(timezone.utc).isoformat()
 receipt={'url':plan['exact_URL'],'requested_at_utc':started,'received_at_utc':received,'curl_exit_code':completed.returncode,'curl_stderr':completed.stderr[:1000],'response_headers':headers.read_text(errors='replace') if headers.exists() else '', 'bytes':partial.stat().st_size if partial.exists() else 0,'partial_local_path':str(partial.relative_to(root))}
 (card/'repair_capture_receipt.json').write_text(json.dumps(receipt,indent=2)+'\n')
 if completed.returncode:raise RuntimeError('single amended ADS ZIP GET failed; preserve partial and do not retry')
 assert partial.stat().st_size==plan['expected_content_length'] and partial.stat().st_size<=plan['budget']['max_response_bytes']
 assert zipfile.is_zipfile(partial)
 partial.replace(source);raw_sha=hashlib.sha256(source.read_bytes()).hexdigest()
 with zipfile.ZipFile(source) as z:
  members=[]
  for item in z.infolist():
   name=PurePosixPath(item.filename);assert not name.is_absolute() and '..' not in name.parts
   if item.is_dir():continue
   members.append({'name':item.filename,'uncompressed_bytes':item.file_size,'compressed_bytes':item.compress_size,'crc32':item.CRC})
  assert members
 settings=_settings(root);archive=Path(tmp)/'original_source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:out.add(source,arcname='ADS_All_Vintages.zip')
 digest=hashlib.sha256(archive.read_bytes()).hexdigest();size=archive.stat().st_size
 evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest
 restore=['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes','-o','ConnectTimeout=5',settings['ssh']['host'],'tar -xOzf '+repr(remote)+' ADS_All_Vintages.zip']
 restored=subprocess.run(restore,capture_output=True,timeout=300);assert restored.returncode==0 and hashlib.sha256(restored.stdout).hexdigest()==raw_sha and len(restored.stdout)==source.stat().st_size
 result={'status':'official_ADS_all_vintages_ZIP_bytes_archived_schema_preflight','repair_plan_sha256':hashlib.sha256(plan_path.read_bytes()).hexdigest(),'raw_ZIP_sha256':raw_sha,'raw_ZIP_bytes':source.stat().st_size,'members':members,'nas_archive_sha256':digest,'nas_archive_bytes':size,'nas_artifact':evidence,'nas_remote_hash_verified':True,'scoped_raw_ZIP_restore_verified':True,'vintage_dates_qualified':False,'total_ZIP_GET_attempts':2,'market_outcome_reads':0,'Qlib_factor_rows':0,'model_fits':0,'native_accounts':0,'SOXX_goal_achieved':False}
 (card/'source_preflight_result.json').write_text(json.dumps(result,indent=2)+'\n')
 (card/'repair_run_state.json').write_text(json.dumps({'status':'completed_source_bytes_schema_pending','repair_plan_sha256':result['repair_plan_sha256'],'additional_GETs':1,'total_GET_attempts':2,'model_fits':0,'accounts':0},indent=2)+'\n')
 print({'status':result['status'],'raw_ZIP_bytes':result['raw_ZIP_bytes'],'members':[m['name'] for m in members]})

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'status': 'official_ADS_all_vintages_ZIP_bytes_archived_schema_preflight', 'raw_ZIP_bytes': 168969969, 'members': ['ADS_All_Vintages-zip.xlsx']}


## 结论边界

完整 ZIP 和 NAS 仅证明来源字节；内部版本结构与时钟须单独核查，再决定是否能进入 Stage1 信息门。

修复预算下完整原始 ZIP 168,969,969 字节及唯一 XLSX 成员已取得；NAS 远端哈希和限定原字节恢复通过。版本日期与 Stage1 信息仍待审。